# 01 Data preparation

Processing xlsx files from data folder into suitable inputs and generate other input files

In [1]:
import pandas as pd
import numpy as np
import os
import pickle

In [2]:
cell_line ='BC3C'
data_dir = f"/home/jing/Phd_project/project_UCD_blca/blca_publication_OUTPUT/blca_publication_OUTPUT_LINCS/00_outputs_2020_{cell_line}/"
info_dir = data_dir
out_dir = f"/home/jing/Phd_project/project_UCD_blca/blca_publication_OUTPUT/blca_publication_OUTPUT_bmra/blca_publication_OUTPUT_bmra_{cell_line}_oct/00_outputs_2020_{cell_line}/"

#os.makedirs(info_dir, exist_ok = True)
os.makedirs(out_dir, exist_ok = True)

## Modules

Load data about modules and drugs.

In [3]:
os.path.join(info_dir, "ALL_DATA_2020_oct25.xlsx")

'/home/jing/Phd_project/project_UCD_blca/blca_publication_OUTPUT/blca_publication_OUTPUT_LINCS/00_outputs_2020_BC3C/ALL_DATA_2020_oct25.xlsx'

In [4]:
### DATA
### remove whitespaces in names (modules), remove duplicates

modules_df = pd.read_excel(
    os.path.join(info_dir, "ALL_DATA_2020_oct25.xlsx"), sheet_name = "modules", index_col = 0)
# display(modules_df)

selected_modules = modules_df.index.tolist()
print(len(selected_modules), ' - Size after reading')

# remove duplicates
selected_modules = modules_df.index.unique().tolist()
print(len(selected_modules), ' - Size after remove duplicates')

# remove whitespaces in modules' names 
selected_modules = [d.strip() for d in selected_modules]

print('Selected modules list: ', len(selected_modules), selected_modules)


12  - Size after reading
12  - Size after remove duplicates
Selected modules list:  12 ['CDK1_2', 'CDK4_6', 'EGFR', 'PI3K', 'FGFR', 'TOP2A', 'p53', 'Src', 'Estrogen', 'Androgen', 'TGFb', 'SMAD3']


In [5]:
### DATA
### remove whitespaces in names (modules, drugs), remove duplicates
### check the dimensions of the indicator IC50 1 uM = 1000 nM
### copy-paste as values, numbers, no formulas

IC50_df = pd.read_excel(
    os.path.join(info_dir, "ALL_DATA_2020_oct25.xlsx"), sheet_name = "IC50s")
IC50_df.drop(columns=['Unnamed: 3','Unnamed: 4'],inplace=True)

print(len(IC50_df.index), ' - Size after reading')
# display(IC50_df)

# rename
IC50_df = IC50_df.rename(columns = {"IC50, uM": "IC50"})

# manually correcting value IC50, Example  for IOX2 -> 30 nM  
#IC50_df.loc[IC50_df.index == 'IOX2', IC50_df.columns == 'IC50'] = 30/1000

# remove non-selected modules, modules' names with whitespaces or empty
IC50_df = IC50_df[IC50_df.Module.isin(selected_modules)]

print(len(IC50_df.index), ' - Size after remove modules')
# display(IC50_df)

# remove duplicates 
# Considering certain columns is optional. 
# Indexes, including time indexes are ignored.
IC50_df = IC50_df.drop_duplicates()

print(len(IC50_df.index), ' - Size after remove duplicates')
display(IC50_df)

42  - Size after reading
42  - Size after remove modules
42  - Size after remove duplicates


,Drug,Module,IC50
0,flufenamic-acid,Androgen,3.00000
1,nandrolone,Androgen,9.00000
2,oxandrolone,Androgen,190.30000
3,testosterone-enanthate,Androgen,200.00000
4,testosterone-propionate,Androgen,124.00000
5,JNJ-7706621,CDK1_2,0.02700
6,PHA-793887,CDK1_2,0.18000
7,roscovitine,CDK1_2,2.00000
8,alvocidib,CDK4_6,0.12000
9,palbociclib,CDK4_6,0.04500


In [6]:
modules = IC50_df.Module.unique().tolist()

print('IC50_df  modules list: ', len(modules), modules)
print()
print('Selected modules list: ', len(selected_modules), selected_modules)

### CHECK
print()
print('CHECK: ', len(selected_modules),'=?', len(modules))

n_modules = len(modules)


IC50_df  modules list:  12 ['Androgen', 'CDK1_2', 'CDK4_6', 'EGFR', 'Estrogen', 'FGFR', 'PI3K', 'p53', 'TOP2A', 'Src', 'TGFb', 'SMAD3']

Selected modules list:  12 ['CDK1_2', 'CDK4_6', 'EGFR', 'PI3K', 'FGFR', 'TOP2A', 'p53', 'Src', 'Estrogen', 'Androgen', 'TGFb', 'SMAD3']

CHECK:  12 =? 12


In [7]:
drugs = IC50_df.Drug.tolist()
print(len(drugs), ' - Size after reading')

# remove duplicates
drugs = IC50_df.Drug.unique().tolist()
print(len(drugs), ' - Size after remove duplicates')

# remove whitespaces in drugs' names (necessary for some)
drugs = [d.strip() for d in drugs]

# remove duplicates after remove whitespaces
drugs = list(set(drugs))
print(len(drugs), ' - Size after remove duplicates without whitespaces')

print('Drugs list: ', len(drugs), drugs)

n_drugs = len(drugs)

42  - Size after reading
42  - Size after remove duplicates
42  - Size after remove duplicates without whitespaces
Drugs list:  42 ['epirubicin', 'daunorubicin', 'ponatinib', 'serdemetan', 'PHA-793887', 'afatinib', 'PI-103', 'JNJ-7706621', 'SAR405838', 'dasatinib', 'testosterone-enanthate', 'sorafenib', 'NVP-BEZ235', 'idarubicin', 'mitoxantrone', 'gefitinib', 'testosterone-propionate', 'vandetanib', 'dienestrol', 'Agent1', 'lapatinib', 'RITA', 'Agent2', 'estradiol-cypionate', 'AS-605240', 'alvocidib', 'raloxifene', 'nutlin-3', 'LY-294002', 'roscovitine', 'AMG-232', 'nandrolone', 'erlotinib', 'oxandrolone', 'taselisib', 'palbociclib', 'HLI-373', 'flufenamic-acid', 'GDC-0349', 'AZD-8055', 'KU-0063794', 'masitinib']


## L1000 meta data

Get sig_id for selected drugs.

In [8]:
sig_info_df = pd.read_excel(os.path.join(data_dir, f"sig_info_2020_{cell_line}.xlsx"), index_col = 0)

display(sig_info_df)

,cell,plate,time,level_3_samples,samples_number,pert_type,pert_drug,targets,targets_number,dose,dose_float
level_5_sig_id,,,,,,,,,,,
ASG002_BC3C_24H:A03,BC3C,ASG002,24 h,ASG002_BC3C_24H_X1_B35:A03,1,ctl_vehicle,DMSO,DMSO_No_target,0,0 uM,0.00
ASG002_BC3C_24H:A04,BC3C,ASG002,24 h,ASG002_BC3C_24H_X1_B35:A04,1,ctl_vehicle,DMSO,DMSO_No_target,0,0 uM,0.00
ASG002_BC3C_24H:A05,BC3C,ASG002,24 h,ASG002_BC3C_24H_X1_B35:A05,1,ctl_vehicle,DMSO,DMSO_No_target,0,0 uM,0.00
ASG002_BC3C_24H:A06,BC3C,ASG002,24 h,ASG002_BC3C_24H_X1_B35:A06,1,ctl_vehicle,DMSO,DMSO_No_target,0,0 uM,0.00
ASG002_BC3C_24H:J13,BC3C,ASG002,24 h,ASG002_BC3C_24H_X1_B35:J13,1,ctl_vehicle,DMSO,DMSO_No_target,0,0 uM,0.00
...,...,...,...,...,...,...,...,...,...,...,...
MOAR012_BC3C_24H:P20,BC3C,MOAR012,24 h,MOAR012_BC3C_24H_X1_B36:P20,1,trt_cp,BAY-61-3606,NaN,0,3.33 uM,3.33
MOAR012_BC3C_24H:P21,BC3C,MOAR012,24 h,MOAR012_BC3C_24H_X1_B36:P21,1,trt_cp,BAY-61-3606,NaN,0,1.11 uM,1.11
MOAR012_BC3C_24H:P22,BC3C,MOAR012,24 h,MOAR012_BC3C_24H_X1_B36:P22,1,trt_cp,ethaverine,NaN,0,10 uM,10.00


In [9]:
# now filtering so only the required drugs are present
sig_info_df = sig_info_df.loc[sig_info_df.pert_drug.isin(drugs)]

# here's what we have now
display(sig_info_df)

,cell,plate,time,level_3_samples,samples_number,pert_type,pert_drug,targets,targets_number,dose,dose_float
level_5_sig_id,,,,,,,,,,,
ASG002_BC3C_24H:A10,BC3C,ASG002,24 h,ASG002_BC3C_24H_X1_B35:A10,1,trt_cp,taselisib,PIK3CA,1,10 uM,10.00
ASG002_BC3C_24H:A11,BC3C,ASG002,24 h,ASG002_BC3C_24H_X1_B35:A11,1,trt_cp,taselisib,PIK3CA,1,1.11 uM,1.11
ASG002_BC3C_24H:A19,BC3C,ASG002,24 h,ASG002_BC3C_24H_X1_B35:A19,1,trt_cp,AS-605240,PIK3CG,1,10 uM,10.00
ASG002_BC3C_24H:A20,BC3C,ASG002,24 h,ASG002_BC3C_24H_X1_B35:A20,1,trt_cp,AS-605240,PIK3CG,1,1.11 uM,1.11
ASG002_BC3C_24H:A21,BC3C,ASG002,24 h,ASG002_BC3C_24H_X1_B35:A21,1,trt_cp,AS-605240,PIK3CG,1,0.12 uM,0.12
...,...,...,...,...,...,...,...,...,...,...,...
MOAR011_BC3C_24H:C11,BC3C,MOAR011,24 h,MOAR011_BC3C_24H_X1_B36:C11,1,trt_cp,testosterone-enanthate,AR,1,3.33 uM,3.33
MOAR011_BC3C_24H:F07,BC3C,MOAR011,24 h,MOAR011_BC3C_24H_X1_B36:F07,1,trt_cp,serdemetan,MDM2,1,10 uM,10.00
MOAR011_BC3C_24H:F08,BC3C,MOAR011,24 h,MOAR011_BC3C_24H_X1_B36:F08,1,trt_cp,serdemetan,MDM2,1,3.33 uM,3.33


In [10]:
#previous removed
#rows_remove = ['ASG002_BC3C_24H:F04', 'ASG002_BC3C_24H:F05','ASG002_BC3C_24H:L02',
#               'ASG002_BC3C_24H:O24','ASG002_B                         C3C_24H:L07','ASG002_BC3C_24H:L09',
#               'MOAR010_BC3C_24H:D02', 'ASG002_BC3C_24H:F01','MOAR010_BC3C_24H:D01',
#               'ASG002_BC3C_24H:N19','ASG002_BC3C_24H:N21','ASG002_BC3C_24H:N24',
#               'ASG002_BC3C_24H:I19','ASG002_BC3C_24H:I21','ASG002_BC3C_24H:L17',
#               'ASG002_BC3C_24H:M23','MOAR008_BC3C_24H:L03','MOAR010_BC3C_24H:L20','MOAR011_BC3C_24H:J10',
#               'MOAR008_BC3C_24H:L08','MOAR009_BC3C_24H:C10','MOAR010_BC3C_24H:A13','MOAR010_BC3C_24H:A14','MOAR011_BC3C_24H:F09',
#               'ASG002_BC3C_24H:G01','ASG002_BC3C_24H:P20']

In [11]:
rows_remove = ['ASG002_BC3C_24H:G01',
 'ASG002_BC3C_24H:F15',
 'ASG002_BC3C_24H:N19',
 'ASG002_BC3C_24H:N22',
 'ASG002_BC3C_24H:F15',
 #'ASG002_BC3C_24H:I08',
 'ASG002_BC3C_24H:I19',
 'ASG002_BC3C_24H:B12',
 'MOAR008_BC3C_24H:L07',
 'MOAR008_BC3C_24H:L09',
 'ASG002_BC3C_24H:F04',
 'ASG002_BC3C_24H:P17',
 'ASG002_BC3C_24H:O24',
 'MOAR010_BC3C_24H:D02',
 'ASG002_BC3C_24H:L18',
 'MOAR008_BC3C_24H:L03',
 'MOAR010_BC3C_24H:L21']

Manually remove  few inhibition from data set, since it does differ from the other data points.

inhib_to_filter = "PF-03758309"
dose_to_filter = 10

id_to_filter = sig_info_df[
    np.logical_and(
        sig_info_df.pert_drug == inhib_to_filter,
        sig_info_df.dose_float == dose_to_filter,
    )
].index.values

print(id_to_filter)

sig_info_df = sig_info_df[~sig_info_df.index.isin(id_to_filter)]
display(sig_info_df)

inhib_to_filter = "roscovitine"
dose_to_filter = 3.33

id_to_filter = sig_info_df[
    np.logical_and(
        sig_info_df.pert_drug == inhib_to_filter,
        sig_info_df.dose_float == dose_to_filter,
    )
].index.values

print(id_to_filter)

sig_info_df = sig_info_df[~sig_info_df.index.isin(id_to_filter)]
display(sig_info_df)

In [12]:
sig_info_df.drop(rows_remove,inplace=True)

In [13]:
exp_ids = sig_info_df.index.unique().tolist()
print('Experiments ids list: ', len(exp_ids))

n_experiments = len(exp_ids)

Experiments ids list:  104


In [14]:
exp_ids = exp_ids+ ['TGFbRin','SMAD2in']
n_experiments = len(exp_ids)
print('Experiments ids list: ', len(exp_ids))


Experiments ids list:  106


Confirm data by checking the drugs of interest against the filtered L1000 meta data.

In [15]:
print(f"Number of drugs of interest:\t{len(drugs)}")
#print(f'Number of drugs in L1000 data:\t{len(sig_info_df.value_counts("drugs"))}')

#sig_info_df.value_counts("drugs")

Number of drugs of interest:	42


## L1000 data

In [16]:
Data_norm_df = pd.read_excel(os.path.join(data_dir, f"Data_norm_2020_{cell_line}.xlsx"), index_col = 0)
display(Data_norm_df)

,AARS,ABCB6,ABCC5,ABCF1,ABCF3,ABHD4,ABHD6,ABL1,ACAA1,ACAT2,...,ZMIZ1,ZMYM2,ZNF131,ZNF274,ZNF318,ZNF395,ZNF451,ZNF586,ZNF589,ZW10
ASG002_BC3C_24H:A03,-0.191254,-0.055246,0.039596,-0.256266,-0.040419,-0.590523,-0.159396,-0.074319,0.457981,0.409608,...,0.543203,0.494266,-0.011923,-0.225931,0.285054,-0.775246,0.166031,-0.024873,0.238723,0.284204
ASG002_BC3C_24H:A04,-0.265754,-0.317496,0.118696,-0.136665,-0.301569,-0.403023,0.124804,-0.036470,0.311931,0.660457,...,-0.565096,-0.088634,0.122977,-0.047931,0.141804,0.129054,-0.028819,-0.028773,-0.253627,-0.752646
ASG002_BC3C_24H:A05,-0.181954,-0.081597,-0.210304,1.559535,-0.019019,-0.457423,0.071404,0.074080,-0.356119,0.498808,...,0.226104,-0.228034,-0.121023,-0.075331,-0.133146,0.355054,0.022831,-0.084073,0.283123,-0.894896
ASG002_BC3C_24H:A06,0.033446,0.042404,-0.150154,-0.093165,0.053180,-0.053823,0.087704,0.167681,-0.601569,0.383308,...,-0.608596,-0.228835,0.072777,0.082970,-0.570996,2.847754,-0.211670,-0.067273,0.081723,0.338704
ASG002_BC3C_24H:J13,0.204446,0.180704,0.089096,-0.054666,0.053381,0.044877,-0.277396,-0.157419,0.535681,-3.933493,...,-0.318397,0.122265,-0.134323,-0.088931,-0.067996,-0.515847,-0.005069,0.067527,0.002223,0.204904
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
MOAR012_BC3C_24H:P20,0.647151,0.211700,-0.979200,0.597350,-0.375751,0.388300,0.394524,0.120151,-0.166775,-1.129125,...,-1.598475,-0.552750,0.515151,0.120800,0.082675,0.529700,0.383225,-0.207225,2.268450,-1.248500
MOAR012_BC3C_24H:P21,0.171800,0.046300,-0.145550,-0.295150,0.030849,0.420951,0.222075,0.179800,0.274724,-0.423975,...,-1.650575,0.203600,-0.003250,-0.064800,-0.037675,0.076499,0.201825,0.416875,0.287450,-0.971700
MOAR012_BC3C_24H:P22,0.648700,0.058749,-0.031700,0.408249,-0.753950,0.332200,-0.357525,-0.107650,-0.213575,0.074225,...,0.127625,0.031600,0.103250,-0.249600,0.046375,1.486200,0.440325,0.090075,-0.031650,-0.944300
MOAR012_BC3C_24H:P23,0.090499,-0.469300,-0.611800,0.873550,-0.788450,-0.097199,-0.366575,-0.490600,-0.624675,-0.009275,...,0.054676,-0.596050,0.084600,0.444700,0.431375,-0.921501,0.044926,0.716076,-0.000900,-1.106700


In [17]:
Data_norm_df = Data_norm_df[Data_norm_df.index.isin(exp_ids)]

# arrange experiments in same order as in list
Data_norm_df["sort_col"] = Data_norm_df.index.map({val: i for i, val in enumerate(exp_ids)})
Data_norm_df = Data_norm_df.sort_values("sort_col")
Data_norm_df = Data_norm_df.drop("sort_col", axis = 1)

# transpose
Data_norm_df = Data_norm_df.T

display(Data_norm_df)

,ASG002_BC3C_24H:A10,ASG002_BC3C_24H:A11,ASG002_BC3C_24H:A19,ASG002_BC3C_24H:A20,ASG002_BC3C_24H:A21,ASG002_BC3C_24H:B10,ASG002_BC3C_24H:B11,ASG002_BC3C_24H:B14,ASG002_BC3C_24H:B15,ASG002_BC3C_24H:C13,...,MOAR010_BC3C_24H:L20,MOAR011_BC3C_24H:C01,MOAR011_BC3C_24H:C02,MOAR011_BC3C_24H:C03,MOAR011_BC3C_24H:C10,MOAR011_BC3C_24H:C11,MOAR011_BC3C_24H:F07,MOAR011_BC3C_24H:F08,MOAR011_BC3C_24H:F09,MOAR011_BC3C_24H:J10
AARS,-0.496854,0.288446,0.189747,-0.016454,0.080746,0.282346,0.326246,0.303046,0.387546,0.434746,...,-0.012317,0.215100,-0.178100,-0.007000,0.024000,0.007400,0.584300,0.114500,-0.268751,0.031399
ABCB6,-0.658596,-0.142196,-0.075397,-0.383796,-0.199996,-0.074197,0.108804,-0.399196,-0.227496,-0.041146,...,-0.661323,0.127850,0.081150,-0.026850,-0.152851,0.122550,0.006950,-0.037150,-0.014150,-0.029350
ABCC5,-0.080204,0.231996,-0.329354,-0.225204,0.278446,0.034696,0.396596,-0.255704,-0.254154,-0.002004,...,-0.051357,0.197500,0.119350,0.212450,-0.177250,0.138650,-0.238750,0.409550,0.221350,0.568950
ABCF1,0.202535,0.602335,0.403335,0.313134,-0.083265,-0.056365,-0.387216,0.097235,0.540634,-0.262666,...,-0.555990,-0.036825,-0.080325,0.059175,0.307175,0.580075,0.142675,0.077875,0.695325,0.322875
ABCF3,-0.520919,-0.192819,0.001032,-0.096419,0.210881,-0.731118,0.095381,0.313481,-0.078018,-0.257669,...,0.447982,-0.378251,0.003950,0.127300,-0.197401,-0.128351,-0.468050,-0.140650,-0.353450,-0.012850
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZNF395,1.796254,1.773154,0.668354,0.825354,0.350654,0.205954,-0.998346,0.235354,0.230653,0.440854,...,-0.050680,0.300250,-0.265350,-0.380850,-0.330950,-1.844650,0.257250,-1.828550,-1.550550,-0.612350
ZNF451,-0.244519,0.116732,0.058081,-0.178169,-0.080619,-0.139119,0.064681,-0.083019,-0.272119,-0.164119,...,-0.083718,-0.324500,0.080550,-0.114950,-0.337650,-0.024950,0.080650,0.119950,0.044250,-0.151550
ZNF586,0.097627,0.061027,-0.337573,0.098427,-0.338173,0.017627,-0.197423,-0.078473,-0.300473,-0.147773,...,-0.123664,-0.304650,-0.259400,0.163600,-0.428700,-0.069200,-0.561900,0.202100,0.043000,0.073700
ZNF589,0.608573,0.106123,-0.014126,-0.003677,-0.123477,-0.243377,0.044623,-0.021477,0.201823,-0.413227,...,0.007359,-0.315500,-0.013900,-0.035050,1.836400,-0.223300,-0.557600,-0.364600,-0.531500,-0.343400


### Concatenating TGFbRin and SMAD3in LFC2

In [18]:
files_path ='/home/jing/Phd_project/project_UCD_blca/blca_publication_OUTPUT/blca_publication_OUTPUT_bc3c_oct'

In [19]:
counts = pd.read_csv(
    '/home/jing/Phd_project/project_UCD_blca/blca_DATA/blca_DATA_bc3c_oct/gene_count.xls',
    sep='\t',   
    header=0,index_col=0
)
display(counts)

,D01,D02,D03,T01,T02,T03,S01,S02,S03,gene_name,gene_chr,gene_start,gene_end,gene_strand,gene_length,gene_biotype,gene_description,tf_family
gene_id,,,,,,,,,,,,,,,,,,
ENSG00000156508,236599,217166,230137,246465,277179,265535,291410,287477,279305,EEF1A1,6,73515750,73523797,-,5948,protein_coding,eukaryotic translation elongation factor 1 alp...,-
ENSG00000186081,173372,151121,162044,168813,186988,177195,144747,149534,158903,KRT5,12,52514575,52520687,-,4292,protein_coding,keratin 5 [Source:HGNC Symbol;Acc:HGNC:6442],-
ENSG00000080824,147634,114075,138470,120147,137207,134926,93481,97859,87311,HSP90AA1,14,102080738,102139699,-,5248,protein_coding,heat shock protein 90 alpha family class A mem...,-
ENSG00000187134,105703,95969,88389,103744,120824,114376,95104,108179,106387,AKR1C1,10,4963253,4983283,+,8765,protein_coding,aldo-keto reductase family 1 member C1 [Source...,-
ENSG00000196139,93872,97803,90777,109018,125059,116421,101868,96017,98182,AKR1C3,10,5035354,5107686,+,4532,protein_coding,aldo-keto reductase family 1 member C3 [Source...,-
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ENSG00000275063,0,0,0,0,0,0,0,0,0,AC233755.1,KI270726.1,41444,41876,+,351,protein_coding,immunoglobulin heavy variable 4-38-2-like [Sou...,-
ENSG00000275405,0,0,0,0,0,0,0,0,0,RF00003,KI270713.1,21861,22024,-,164,snRNA,NaN,-
ENSG00000275987,0,0,0,0,0,0,0,0,0,RF00003,KI270713.1,30437,30580,-,144,snRNA,NaN,-


In [20]:
tgfb_df = pd.read_csv(os.path.join(files_path,'Group_T_vs_D.csv'),index_col=0)
display(tgfb_df)
smad_df  =pd.read_csv(os.path.join(files_path,'Group_S_vs_D.csv'),index_col=0)
display(smad_df)

,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj
ENSG00000156508,256430.450742,0.166341,0.063865,2.604588,0.009198,0.064874
ENSG00000186081,162335.665169,0.094749,0.069048,1.372219,0.169995,0.426880
ENSG00000080824,117996.596529,-0.061632,0.057216,-1.077179,0.281400,0.559577
ENSG00000187134,103306.722000,0.184925,0.074447,2.483981,0.012992,0.082667
ENSG00000196139,102475.740567,0.264850,0.084255,3.143448,0.001670,0.017763
...,...,...,...,...,...,...
ENSG00000187045,1.108841,0.569208,1.970043,0.288932,0.772634,NaN
ENSG00000270022,1.109888,-3.236412,2.028046,-1.595828,0.110527,NaN
ENSG00000227698,1.122733,-2.255838,2.083268,-1.082836,0.278881,NaN
ENSG00000273027,1.080752,-1.458468,2.036725,-0.716085,0.473939,NaN


,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj
ENSG00000156508,256430.450742,0.259346,0.063863,4.060979,4.886738e-05,3.454193e-04
ENSG00000186081,162335.665169,-0.166092,0.069053,-2.405280,1.616007e-02,5.227679e-02
ENSG00000080824,117996.596529,-0.584208,0.057235,-10.207122,1.842492e-24,1.722188e-22
ENSG00000187134,103306.722000,0.027286,0.074451,0.366501,7.139914e-01,8.322319e-01
ENSG00000196139,102475.740567,-0.005072,0.084261,-0.060192,9.520026e-01,9.758303e-01
...,...,...,...,...,...,...
ENSG00000187045,1.108841,1.243363,1.902327,0.653601,5.133689e-01,NaN
ENSG00000270022,1.109888,-0.096033,1.773277,-0.054156,9.568110e-01,NaN
ENSG00000227698,1.122733,-0.342667,1.859081,-0.184321,8.537620e-01,NaN
ENSG00000273027,1.080752,0.992968,1.762833,0.563280,5.732445e-01,NaN


In [21]:
tgfb_df['symbol']= counts.loc[tgfb_df.index,'gene_name']
display(tgfb_df)
smad_df['symbol']= counts.loc[smad_df.index,'gene_name']
display(smad_df)

,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj,symbol
ENSG00000156508,256430.450742,0.166341,0.063865,2.604588,0.009198,0.064874,EEF1A1
ENSG00000186081,162335.665169,0.094749,0.069048,1.372219,0.169995,0.426880,KRT5
ENSG00000080824,117996.596529,-0.061632,0.057216,-1.077179,0.281400,0.559577,HSP90AA1
ENSG00000187134,103306.722000,0.184925,0.074447,2.483981,0.012992,0.082667,AKR1C1
ENSG00000196139,102475.740567,0.264850,0.084255,3.143448,0.001670,0.017763,AKR1C3
...,...,...,...,...,...,...,...
ENSG00000187045,1.108841,0.569208,1.970043,0.288932,0.772634,NaN,TMPRSS6
ENSG00000270022,1.109888,-3.236412,2.028046,-1.595828,0.110527,NaN,Z93241.1
ENSG00000227698,1.122733,-2.255838,2.083268,-1.082836,0.278881,NaN,AP001619.1
ENSG00000273027,1.080752,-1.458468,2.036725,-0.716085,0.473939,NaN,AL844908.2


,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj,symbol
ENSG00000156508,256430.450742,0.259346,0.063863,4.060979,4.886738e-05,3.454193e-04,EEF1A1
ENSG00000186081,162335.665169,-0.166092,0.069053,-2.405280,1.616007e-02,5.227679e-02,KRT5
ENSG00000080824,117996.596529,-0.584208,0.057235,-10.207122,1.842492e-24,1.722188e-22,HSP90AA1
ENSG00000187134,103306.722000,0.027286,0.074451,0.366501,7.139914e-01,8.322319e-01,AKR1C1
ENSG00000196139,102475.740567,-0.005072,0.084261,-0.060192,9.520026e-01,9.758303e-01,AKR1C3
...,...,...,...,...,...,...,...
ENSG00000187045,1.108841,1.243363,1.902327,0.653601,5.133689e-01,NaN,TMPRSS6
ENSG00000270022,1.109888,-0.096033,1.773277,-0.054156,9.568110e-01,NaN,Z93241.1
ENSG00000227698,1.122733,-0.342667,1.859081,-0.184321,8.537620e-01,NaN,AP001619.1
ENSG00000273027,1.080752,0.992968,1.762833,0.563280,5.732445e-01,NaN,AL844908.2


In [22]:
tgfb_lfc = tgfb_df.set_index('symbol')
tgfb_lfc = tgfb_lfc.loc[tgfb_lfc.index.intersection(Data_norm_df.index)]
display(tgfb_lfc)
smad_lfc = smad_df.set_index('symbol')
smad_lfc = smad_lfc.loc[smad_lfc.index.intersection(Data_norm_df.index)]
display(smad_lfc)

,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj
GAPDH,67670.120631,0.085744,0.069671,1.230699,2.184353e-01,4.894558e-01
HSPD1,43977.253750,0.001495,0.055334,0.027014,9.784487e-01,9.918190e-01
HSPA8,35918.997223,-0.498890,0.062210,-8.019419,1.062467e-15,1.332019e-13
SPP1,33829.983939,0.140098,0.059159,2.368159,1.787686e-02,1.038853e-01
TXNRD1,31108.470837,0.329729,0.070416,4.682587,2.832769e-06,8.109026e-05
...,...,...,...,...,...,...
STAP2,3.106633,0.377292,1.166433,0.323458,7.463483e-01,NaN
PRR15L,2.549797,-0.415355,1.205695,-0.344494,7.304748e-01,NaN
PTPRC,2.168900,-1.182366,1.262117,-0.936812,3.488554e-01,NaN
TBXA2R,1.540184,-0.649463,1.773400,-0.366225,7.141972e-01,NaN


,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj
GAPDH,67670.120631,-0.299123,0.069689,-4.292246,1.768744e-05,1.394115e-04
HSPD1,43977.253750,-0.622301,0.055397,-11.233402,2.795038e-29,4.037559e-27
HSPA8,35918.997223,-0.516719,0.062209,-8.306214,9.880391e-17,4.313171e-15
SPP1,33829.983939,0.158182,0.059154,2.674079,7.493480e-03,2.758922e-02
TXNRD1,31108.470837,-0.297777,0.070481,-4.224905,2.390416e-05,1.827019e-04
...,...,...,...,...,...,...
STAP2,3.106633,0.054051,1.184627,0.045627,9.636072e-01,NaN
PRR15L,2.549797,-1.486121,1.311748,-1.132932,2.572427e-01,NaN
PTPRC,2.168900,-4.655504,1.622841,-2.868738,4.121133e-03,NaN
TBXA2R,1.540184,-0.617323,1.767028,-0.349357,7.268215e-01,NaN


In [23]:
Data_norm_df['TGFbRin'] = 0
Data_norm_df['SMAD2in'] = 0

for i in tgfb_lfc.index.intersection(Data_norm_df.index):
    Data_norm_df.loc[i,'TGFbRin'] =tgfb_lfc.loc[i,'log2FoldChange']


for i in smad_lfc.index.intersection(Data_norm_df.index):
    Data_norm_df.loc[i,'SMAD2in'] =smad_lfc.loc[i,'log2FoldChange']


/tmp/ipykernel_3921679/3696558090.py:5: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.0857437848539165' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  Data_norm_df.loc[i,'TGFbRin'] =tgfb_lfc.loc[i,'log2FoldChange']
/tmp/ipykernel_3921679/3696558090.py:9: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '-0.29912277051591' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  Data_norm_df.loc[i,'SMAD2in'] =smad_lfc.loc[i,'log2FoldChange']


In [24]:
Data_norm_df

,ASG002_BC3C_24H:A10,ASG002_BC3C_24H:A11,ASG002_BC3C_24H:A19,ASG002_BC3C_24H:A20,ASG002_BC3C_24H:A21,ASG002_BC3C_24H:B10,ASG002_BC3C_24H:B11,ASG002_BC3C_24H:B14,ASG002_BC3C_24H:B15,ASG002_BC3C_24H:C13,...,MOAR011_BC3C_24H:C02,MOAR011_BC3C_24H:C03,MOAR011_BC3C_24H:C10,MOAR011_BC3C_24H:C11,MOAR011_BC3C_24H:F07,MOAR011_BC3C_24H:F08,MOAR011_BC3C_24H:F09,MOAR011_BC3C_24H:J10,TGFbRin,SMAD2in
AARS,-0.496854,0.288446,0.189747,-0.016454,0.080746,0.282346,0.326246,0.303046,0.387546,0.434746,...,-0.178100,-0.007000,0.024000,0.007400,0.584300,0.114500,-0.268751,0.031399,0.486026,0.739698
ABCB6,-0.658596,-0.142196,-0.075397,-0.383796,-0.199996,-0.074197,0.108804,-0.399196,-0.227496,-0.041146,...,0.081150,-0.026850,-0.152851,0.122550,0.006950,-0.037150,-0.014150,-0.029350,0.397480,0.014959
ABCC5,-0.080204,0.231996,-0.329354,-0.225204,0.278446,0.034696,0.396596,-0.255704,-0.254154,-0.002004,...,0.119350,0.212450,-0.177250,0.138650,-0.238750,0.409550,0.221350,0.568950,0.311687,-0.105603
ABCF1,0.202535,0.602335,0.403335,0.313134,-0.083265,-0.056365,-0.387216,0.097235,0.540634,-0.262666,...,-0.080325,0.059175,0.307175,0.580075,0.142675,0.077875,0.695325,0.322875,-0.049201,-0.282741
ABCF3,-0.520919,-0.192819,0.001032,-0.096419,0.210881,-0.731118,0.095381,0.313481,-0.078018,-0.257669,...,0.003950,0.127300,-0.197401,-0.128351,-0.468050,-0.140650,-0.353450,-0.012850,0.154353,-0.029281
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZNF395,1.796254,1.773154,0.668354,0.825354,0.350654,0.205954,-0.998346,0.235354,0.230653,0.440854,...,-0.265350,-0.380850,-0.330950,-1.844650,0.257250,-1.828550,-1.550550,-0.612350,0.448519,0.004234
ZNF451,-0.244519,0.116732,0.058081,-0.178169,-0.080619,-0.139119,0.064681,-0.083019,-0.272119,-0.164119,...,0.080550,-0.114950,-0.337650,-0.024950,0.080650,0.119950,0.044250,-0.151550,-0.116411,-0.058570
ZNF586,0.097627,0.061027,-0.337573,0.098427,-0.338173,0.017627,-0.197423,-0.078473,-0.300473,-0.147773,...,-0.259400,0.163600,-0.428700,-0.069200,-0.561900,0.202100,0.043000,0.073700,0.015062,-0.364378
ZNF589,0.608573,0.106123,-0.014126,-0.003677,-0.123477,-0.243377,0.044623,-0.021477,0.201823,-0.413227,...,-0.013900,-0.035050,1.836400,-0.223300,-0.557600,-0.364600,-0.531500,-0.343400,0.002455,-0.173722


In [25]:
genes = Data_norm_df.index.tolist()
print('Landmark genes list: ', len(genes), genes)

n_genes = len(genes)

Landmark genes list:  978 ['AARS', 'ABCB6', 'ABCC5', 'ABCF1', 'ABCF3', 'ABHD4', 'ABHD6', 'ABL1', 'ACAA1', 'ACAT2', 'ACBD3', 'ACD', 'ACLY', 'ACOT9', 'ADAM10', 'ADAT1', 'ADGRE5', 'ADGRG1', 'ADH5', 'ADI1', 'ADO', 'ADRB2', 'AGL', 'AKAP8', 'AKAP8L', 'AKR7A2', 'AKT1', 'ALAS1', 'ALDH7A1', 'ALDOA', 'ALDOC', 'AMDHD2', 'ANKRD10', 'ANO10', 'ANXA7', 'APBB2', 'APOE', 'APP', 'APPBP2', 'ARFIP2', 'ARHGAP1', 'ARHGEF12', 'ARHGEF2', 'ARID4B', 'ARID5B', 'ARL4C', 'ARNT2', 'ARPP19', 'ASAH1', 'ASCC3', 'ATF1', 'ATF5', 'ATF6', 'ATG3', 'ATMIN', 'ATP11B', 'ATP1B1', 'ATP2C1', 'ATP6V0B', 'ATP6V1D', 'AURKA', 'AURKB', 'AXIN1', 'B4GAT1', 'BACE2', 'BAD', 'BAG3', 'BAMBI', 'BAX', 'BCL2', 'BCL7B', 'BDH1', 'BECN1', 'BHLHE40', 'BID', 'BIRC2', 'BIRC5', 'BLCAP', 'BLMH', 'BLVRA', 'BMP4', 'BNIP3', 'BNIP3L', 'BPHL', 'BRCA1', 'BTK', 'BUB1B', 'BZW2', 'C2CD2', 'C2CD2L', 'C2CD5', 'C5', 'CAB39', 'CALM3', 'CALU', 'CAMSAP2', 'CANT1', 'CAPN1', 'CARMIL1', 'CASC3', 'CASK', 'CASP10', 'CASP2', 'CASP3', 'CASP7', 'CAST', 'CAT', 'CBLB', 'CBR1

## Inhibitor concentrations, IC50, and perturbation matrices

In [26]:
inhib_conc_matrix = np.zeros((n_modules, n_experiments))
ic50_matrix = np.ones((n_modules, n_experiments))
gamma_matrix = np.zeros((n_modules, n_experiments))

In [27]:

for i, module in enumerate(modules):
    drugs_for_module = IC50_df.Drug[IC50_df.Module == module].tolist()
    for drug in drugs_for_module:
        # get IC50 for this drug
        ic50 = IC50_df.IC50[IC50_df.Drug == drug].values
#       gamma = IC50_df.Gamma[IC50_df.Drug == drug].values
        print(drug, ic50)
        assert ic50.size == 1
#       assert gamma.size == 1
        # get experiments with this drug
        exp_with_drug = sig_info_df.index[sig_info_df.pert_drug == drug].tolist()
        print(exp_with_drug) 
        for exp_id in exp_with_drug:
            j = exp_ids.index(exp_id)
            print(j)
            # extract inhibitor concentration
            inhib_conc = sig_info_df.dose_float[sig_info_df.index == exp_id].values
            assert inhib_conc.size == 1
            # insert values in matrices
            inhib_conc_matrix[i, j] = inhib_conc.item()
            ic50_matrix[i, j] = ic50.item()
#           gamma_matrix[i, j] = gamma.item()


flufenamic-acid [3.]
['MOAR008_BC3C_24H:L01', 'MOAR008_BC3C_24H:L02']
76
77
nandrolone [9.]
['MOAR011_BC3C_24H:J10']
103
oxandrolone [190.3]
['MOAR011_BC3C_24H:C01', 'MOAR011_BC3C_24H:C02', 'MOAR011_BC3C_24H:C03']
95
96
97
testosterone-enanthate [200.]
['MOAR011_BC3C_24H:C10', 'MOAR011_BC3C_24H:C11']
98
99
testosterone-propionate [124.]
['MOAR010_BC3C_24H:L19', 'MOAR010_BC3C_24H:L20']
93
94
JNJ-7706621 [0.027]
['ASG002_BC3C_24H:N13', 'ASG002_BC3C_24H:N14', 'ASG002_BC3C_24H:N15']
57
58
59
PHA-793887 [0.18]
['ASG002_BC3C_24H:L01', 'ASG002_BC3C_24H:L02', 'ASG002_BC3C_24H:L03']
46
47
48
roscovitine [2.]
['ASG002_BC3C_24H:E22', 'ASG002_BC3C_24H:E23', 'ASG002_BC3C_24H:E24']
18
19
20
alvocidib [0.12]
['ASG002_BC3C_24H:F05', 'ASG002_BC3C_24H:F06']
24
25
palbociclib [0.045]
['ASG002_BC3C_24H:P16', 'ASG002_BC3C_24H:P18']
72
73
afatinib [0.03]
['ASG002_BC3C_24H:N23', 'ASG002_BC3C_24H:N24']
62
63
erlotinib [0.006]
['ASG002_BC3C_24H:H16', 'ASG002_BC3C_24H:H17', 'ASG002_BC3C_24H:H18']
33
34
35
gefit

In [28]:
# transform matrices into pandas dfs for export with row and column names
inhib_conc_df = pd.DataFrame(inhib_conc_matrix, index = modules, columns = exp_ids)
ic50_df = pd.DataFrame(ic50_matrix, index = modules, columns = exp_ids)
# gamma_df = pd.DataFrame(gamma_matrix, index = modules, columns = exp_ids)

# create binary perturbation matrix
pert_df = pd.DataFrame(
    np.where(inhib_conc_matrix != 0, 1, 0),
    index = inhib_conc_df.index,
    columns = inhib_conc_df.columns,
)

In [29]:
display(ic50_df)
# display(gamma_df)
display(inhib_conc_df)
display(pert_df)

,ASG002_BC3C_24H:A10,ASG002_BC3C_24H:A11,ASG002_BC3C_24H:A19,ASG002_BC3C_24H:A20,ASG002_BC3C_24H:A21,ASG002_BC3C_24H:B10,ASG002_BC3C_24H:B11,ASG002_BC3C_24H:B14,ASG002_BC3C_24H:B15,ASG002_BC3C_24H:C13,...,MOAR011_BC3C_24H:C02,MOAR011_BC3C_24H:C03,MOAR011_BC3C_24H:C10,MOAR011_BC3C_24H:C11,MOAR011_BC3C_24H:F07,MOAR011_BC3C_24H:F08,MOAR011_BC3C_24H:F09,MOAR011_BC3C_24H:J10,TGFbRin,SMAD2in
Androgen,1.00000,1.00000,1.0000,1.0000,1.0000,1.0000,1.0000,1.00,1.00,1.00,...,190.3,190.3,200.0,200.0,1.00,1.00,1.00,9.0,1.0,1.0
CDK1_2,1.00000,1.00000,1.0000,1.0000,1.0000,1.0000,1.0000,1.00,1.00,1.00,...,1.0,1.0,1.0,1.0,1.00,1.00,1.00,1.0,1.0,1.0
CDK4_6,1.00000,1.00000,1.0000,1.0000,1.0000,1.0000,1.0000,1.00,1.00,1.00,...,1.0,1.0,1.0,1.0,1.00,1.00,1.00,1.0,1.0,1.0
EGFR,1.00000,1.00000,1.0000,1.0000,1.0000,1.0000,1.0000,1.00,1.00,1.00,...,1.0,1.0,1.0,1.0,1.00,1.00,1.00,1.0,1.0,1.0
Estrogen,1.00000,1.00000,1.0000,1.0000,1.0000,1.0000,1.0000,1.00,1.00,1.00,...,1.0,1.0,1.0,1.0,1.00,1.00,1.00,1.0,1.0,1.0
FGFR,1.00000,1.00000,1.0000,1.0000,1.0000,1.0000,1.0000,1.00,1.00,1.00,...,1.0,1.0,1.0,1.0,1.00,1.00,1.00,1.0,1.0,1.0
PI3K,0.00262,0.00262,0.1595,0.1595,0.1595,1.0000,1.0000,1.00,1.00,1.00,...,1.0,1.0,1.0,1.0,1.00,1.00,1.00,1.0,1.0,1.0
p53,1.00000,1.00000,1.0000,1.0000,1.0000,0.0018,0.0018,9.75,9.75,0.54,...,1.0,1.0,1.0,1.0,62.15,62.15,62.15,1.0,1.0,1.0
TOP2A,1.00000,1.00000,1.0000,1.0000,1.0000,1.0000,1.0000,1.00,1.00,1.00,...,1.0,1.0,1.0,1.0,1.00,1.00,1.00,1.0,1.0,1.0
Src,1.00000,1.00000,1.0000,1.0000,1.0000,1.0000,1.0000,1.00,1.00,1.00,...,1.0,1.0,1.0,1.0,1.00,1.00,1.00,1.0,1.0,1.0


,ASG002_BC3C_24H:A10,ASG002_BC3C_24H:A11,ASG002_BC3C_24H:A19,ASG002_BC3C_24H:A20,ASG002_BC3C_24H:A21,ASG002_BC3C_24H:B10,ASG002_BC3C_24H:B11,ASG002_BC3C_24H:B14,ASG002_BC3C_24H:B15,ASG002_BC3C_24H:C13,...,MOAR011_BC3C_24H:C02,MOAR011_BC3C_24H:C03,MOAR011_BC3C_24H:C10,MOAR011_BC3C_24H:C11,MOAR011_BC3C_24H:F07,MOAR011_BC3C_24H:F08,MOAR011_BC3C_24H:F09,MOAR011_BC3C_24H:J10,TGFbRin,SMAD2in
Androgen,0.0,0.00,0.0,0.00,0.00,0.0,0.00,0.00,0.00,0.0,...,3.33,1.11,10.0,3.33,0.0,0.00,0.00,10.0,0.0,0.0
CDK1_2,0.0,0.00,0.0,0.00,0.00,0.0,0.00,0.00,0.00,0.0,...,0.00,0.00,0.0,0.00,0.0,0.00,0.00,0.0,0.0,0.0
CDK4_6,0.0,0.00,0.0,0.00,0.00,0.0,0.00,0.00,0.00,0.0,...,0.00,0.00,0.0,0.00,0.0,0.00,0.00,0.0,0.0,0.0
EGFR,0.0,0.00,0.0,0.00,0.00,0.0,0.00,0.00,0.00,0.0,...,0.00,0.00,0.0,0.00,0.0,0.00,0.00,0.0,0.0,0.0
Estrogen,0.0,0.00,0.0,0.00,0.00,0.0,0.00,0.00,0.00,0.0,...,0.00,0.00,0.0,0.00,0.0,0.00,0.00,0.0,0.0,0.0
FGFR,0.0,0.00,0.0,0.00,0.00,0.0,0.00,0.00,0.00,0.0,...,0.00,0.00,0.0,0.00,0.0,0.00,0.00,0.0,0.0,0.0
PI3K,10.0,1.11,10.0,1.11,0.12,0.0,0.00,0.00,0.00,0.0,...,0.00,0.00,0.0,0.00,0.0,0.00,0.00,0.0,0.0,0.0
p53,0.0,0.00,0.0,0.00,0.00,10.0,1.11,1.11,0.08,10.0,...,0.00,0.00,0.0,0.00,10.0,3.33,1.11,0.0,0.0,0.0
TOP2A,0.0,0.00,0.0,0.00,0.00,0.0,0.00,0.00,0.00,0.0,...,0.00,0.00,0.0,0.00,0.0,0.00,0.00,0.0,0.0,0.0
Src,0.0,0.00,0.0,0.00,0.00,0.0,0.00,0.00,0.00,0.0,...,0.00,0.00,0.0,0.00,0.0,0.00,0.00,0.0,0.0,0.0


,ASG002_BC3C_24H:A10,ASG002_BC3C_24H:A11,ASG002_BC3C_24H:A19,ASG002_BC3C_24H:A20,ASG002_BC3C_24H:A21,ASG002_BC3C_24H:B10,ASG002_BC3C_24H:B11,ASG002_BC3C_24H:B14,ASG002_BC3C_24H:B15,ASG002_BC3C_24H:C13,...,MOAR011_BC3C_24H:C02,MOAR011_BC3C_24H:C03,MOAR011_BC3C_24H:C10,MOAR011_BC3C_24H:C11,MOAR011_BC3C_24H:F07,MOAR011_BC3C_24H:F08,MOAR011_BC3C_24H:F09,MOAR011_BC3C_24H:J10,TGFbRin,SMAD2in
Androgen,0,0,0,0,0,0,0,0,0,0,...,1,1,1,1,0,0,0,1,0,0
CDK1_2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
CDK4_6,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
EGFR,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Estrogen,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
FGFR,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
PI3K,1,1,1,1,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
p53,0,0,0,0,0,1,1,1,1,1,...,0,0,0,0,1,1,1,0,0,0
TOP2A,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Src,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## Global responses for DPD modules

According to our discussion, $R$ for DPD vectors can not be calculated with the same formula as for pathway activities. Instead we are assuming:

\begin{equation}
R_{DPD},j = DPD = STV_{DPD} \cdot Data_j
\end{equation}

In [30]:
# load STV data frame
STVs = pd.read_excel(os.path.join(info_dir, "ALL_DATA_2020_july_25.xlsx"), sheet_name = "STV", index_col = 0)
STV_df = pd.DataFrame(np.zeros((len(Data_norm_df.index), 3)), index = Data_norm_df.index, columns = STVs.columns)
STV_df.loc[STVs.index] = STVs

display(STV_df)

,blca_basal_luminal,blca_oncogenesis,blca_survival
AARS,0.0,0.078487,0.000000
ABCB6,0.0,0.027125,-2.890931
ABCC5,0.0,0.002121,0.000000
ABCF1,0.0,-0.028959,-0.496186
ABCF3,0.0,0.036671,0.000000
...,...,...,...
ZNF395,0.0,0.000000,0.000000
ZNF451,0.0,0.013403,0.000000
ZNF586,0.0,0.000000,0.119103
ZNF589,0.0,0.000000,0.000000


In [31]:
# create empty DPD data frame
DPD_df = pd.DataFrame(
    np.zeros((len(Data_norm_df.columns), len(STV_df.columns))),
    index = Data_norm_df.columns,
    columns = STV_df.columns,
)

# populate
for exp_id in DPD_df.index:
    for state in STV_df.columns:
        DPD_df.loc[exp_id, state] = np.dot(Data_norm_df.T.loc[exp_id], STV_df.loc[:, state])

display(DPD_df)

,blca_basal_luminal,blca_oncogenesis,blca_survival
ASG002_BC3C_24H:A10,-56.563525,-0.610650,-17.377163
ASG002_BC3C_24H:A11,-51.696023,-0.936049,-5.764659
ASG002_BC3C_24H:A19,-7.483112,-0.048496,9.872480
ASG002_BC3C_24H:A20,3.688500,-0.647673,-0.584775
ASG002_BC3C_24H:A21,-12.723954,0.031650,-6.200738
...,...,...,...
MOAR011_BC3C_24H:F08,-10.680550,0.769415,12.596146
MOAR011_BC3C_24H:F09,-11.391052,0.153872,7.521684
MOAR011_BC3C_24H:J10,-6.426323,-0.115337,5.555040
TGFbRin,-12.272085,-0.129464,-10.057353


In [32]:
# transform to R global
R_global_DPD_df = DPD_df.T
display(R_global_DPD_df)

,ASG002_BC3C_24H:A10,ASG002_BC3C_24H:A11,ASG002_BC3C_24H:A19,ASG002_BC3C_24H:A20,ASG002_BC3C_24H:A21,ASG002_BC3C_24H:B10,ASG002_BC3C_24H:B11,ASG002_BC3C_24H:B14,ASG002_BC3C_24H:B15,ASG002_BC3C_24H:C13,...,MOAR011_BC3C_24H:C02,MOAR011_BC3C_24H:C03,MOAR011_BC3C_24H:C10,MOAR011_BC3C_24H:C11,MOAR011_BC3C_24H:F07,MOAR011_BC3C_24H:F08,MOAR011_BC3C_24H:F09,MOAR011_BC3C_24H:J10,TGFbRin,SMAD2in
blca_basal_luminal,-56.563525,-51.696023,-7.483112,3.688500,-12.723954,-14.104169,4.258746,-26.565452,-5.792472,-15.570078,...,-10.048598,-3.096432,-9.738229,-9.099212,-52.623843,-10.680550,-11.391052,-6.426323,-12.272085,-75.253190
blca_oncogenesis,-0.610650,-0.936049,-0.048496,-0.647673,0.031650,0.185978,0.911126,-0.079387,-0.459741,-0.521443,...,0.106842,-0.068887,-0.875167,-0.261326,0.259917,0.769415,0.153872,-0.115337,-0.129464,-0.939899
blca_survival,-17.377163,-5.764659,9.872480,-0.584775,-6.200738,1.836483,-3.060844,-3.581400,1.041249,-6.062045,...,1.064820,5.703725,9.754974,-4.383434,6.540675,12.596146,7.521684,5.555040,-10.057353,-15.346702


## Save outputs

In [33]:
# save metadata as pickle
all_metadata = {
    "modules": modules,
    "n_modules": n_modules,
    "drugs": drugs,
    "n_drugs": n_drugs,
    "exp_ids": exp_ids,
    "n_experiments": n_experiments,
    "genes": genes,
    "n_genes": n_genes,
}

print(all_metadata)

with open(os.path.join(out_dir, "metadata.pickle"), "wb") as f:
    pickle.dump(all_metadata, f, protocol = pickle.HIGHEST_PROTOCOL)

{'modules': ['Androgen', 'CDK1_2', 'CDK4_6', 'EGFR', 'Estrogen', 'FGFR', 'PI3K', 'p53', 'TOP2A', 'Src', 'TGFb', 'SMAD3'], 'n_modules': 12, 'drugs': ['epirubicin', 'daunorubicin', 'ponatinib', 'serdemetan', 'PHA-793887', 'afatinib', 'PI-103', 'JNJ-7706621', 'SAR405838', 'dasatinib', 'testosterone-enanthate', 'sorafenib', 'NVP-BEZ235', 'idarubicin', 'mitoxantrone', 'gefitinib', 'testosterone-propionate', 'vandetanib', 'dienestrol', 'Agent1', 'lapatinib', 'RITA', 'Agent2', 'estradiol-cypionate', 'AS-605240', 'alvocidib', 'raloxifene', 'nutlin-3', 'LY-294002', 'roscovitine', 'AMG-232', 'nandrolone', 'erlotinib', 'oxandrolone', 'taselisib', 'palbociclib', 'HLI-373', 'flufenamic-acid', 'GDC-0349', 'AZD-8055', 'KU-0063794', 'masitinib'], 'n_drugs': 42, 'exp_ids': ['ASG002_BC3C_24H:A10', 'ASG002_BC3C_24H:A11', 'ASG002_BC3C_24H:A19', 'ASG002_BC3C_24H:A20', 'ASG002_BC3C_24H:A21', 'ASG002_BC3C_24H:B10', 'ASG002_BC3C_24H:B11', 'ASG002_BC3C_24H:B14', 'ASG002_BC3C_24H:B15', 'ASG002_BC3C_24H:C13', 'A

In [34]:
# save doses and perturbation matrix
inhib_conc_df.to_csv(os.path.join(out_dir, "inhib_conc_annotated.csv"))
ic50_df.to_csv(os.path.join(out_dir, "ic50_annotated.csv"))
# gamma_df.to_csv(os.path.join(out_dir, "gamma_annotated.csv"))
pert_df.to_csv(os.path.join(out_dir, "pert_annotated.csv"))

In [35]:
# save log fold change L1000 data
Data_norm_df.to_csv(os.path.join(out_dir, "L1000_Data_norm_data.csv"))

In [36]:
# save R_global for DPDs
R_global_DPD_df.to_csv(os.path.join(out_dir, "R_global_DPDonly_annotated.csv"))